# Main Quest 1. Unsupervised Feature Discovery: Building the Foundational Decoder Model

In [ ]:
# ===========================================================================
# Global Imports & Configuration
# ===========================================================================
# 1. Essential Libraries 
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader, random_split
import torch.optim.lr_scheduler as lr_scheduler

import os, glob
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import re
import logging

from utils import *
from models import * 
# -------------------------------------------------------------------------
# Set device configuration
# -------------------------------------------------------------------------
if torch.backends.mps.is_available():
    device = torch.device("mps")
    print(">>> Using Apple Silicon (MPS) for acceleration.")
else:
    device = torch.device("cpu")
    print(">>> Using CPU.")

# -------------------------------------------------------------------------
# 0. Define Hyperparameters (MUST match Stage I Pretraining)
# -------------------------------------------------------------------------
HP = {
    "vocab_size": 8000, 
    "max_len": 128,
    "d_model": 256,
    "num_layers": 4,
    "num_heads": 8,
    "d_ff": 512,
    "dropout": 0.1,
    "learning_rate": 5e-5,    
    "batch_size": 64,       # Pre-training usually benefits from larger batches
    "patience": 10, 
    "warmup_steps": 1000,   # <<< Pre-training Standard: 4000
    "label_smoothing" : 0.1 
}
# -------------------------------------------------------------------------
# 1. Initialize the Model Shell
# -------------------------------------------------------------------------
model = GPTModel(
    vocab_size=HP["vocab_size"],
    d_model=HP["d_model"],
    num_layers=HP["num_layers"],
    num_heads=HP["num_heads"],
    ff_dim=HP["d_ff"],
    max_len=HP["max_len"],
    dropout=HP["dropout"]
).to(device)

In [ ]:
# -------------------------------------------------------------------------
# STEP 2: Load Raw Text Corpus
# -------------------------------------------------------------------------
# We load the dataset with NO transform initially to keep the raw data pure.
import urllib.request
data_path = './data'

# Ensure data directory exists
os.makedirs(data_path, exist_ok=True)

file_path = os.path.join(data_path, "ChatbotData.csv")

# Download dataset if it does not exist
if not os.path.exists(file_path):
    print(">>> Dataset not found. Downloading now...")
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/songys/Chatbot_data/master/ChatbotData.csv",
        filename=file_path
    )
    print(">>> Download completed and saved to:", file_path)
else:
    print(">>> Dataset already exists. Skipping download.")

# Load dataset
raw_data = pd.read_csv(file_path,  encoding='utf-8-sig')

print(f">>> Successfully loaded dataset with {len(raw_data)} rows.")
print(f">>> Column Names: {list(raw_data.columns)}")

In [ ]:
# ===========================================================================
# STEP 2-2: Constructing the Pre-training Corpus
# ===========================================================================
# Instead of treating them as isolated Q&A pairs, we view this as a continuous 
# linguistic stream. We maintain the [Q, A] structure to bake the conversational 
# syntax into the model's weights during the Unsupervised phase.
data = raw_data[['Q', 'A']].copy()
data.columns = ['question', 'answer']

# ===========================================================================
# STEP 2-3: Data Integrity & Quality Control (NaN Filtering)
# ===========================================================================
# Ensuring high-quality training signals is paramount for CLM. We remove 
# incomplete entries to avoid injecting noise into the Next-Token Prediction 
# objective, ensuring the model learns only from valid semantic pairs.
data.dropna(inplace=True)

# ===========================================================================
# STEP 2-4: Token-Level Normalization (Denoising)
# ===========================================================================
# We apply rigorous whitespace normalization. This minimizes variance in 
# subword segmentation, ensuring the Learned Positional Embeddings (W_pos) 
# and Attention heads focus on semantic tokens rather than formatting noise.
def clean_text(text):
    # Denoising: Stripping edges and collapsing redundant whitespace
    text = str(text).strip()
    text = re.sub(r'\s+', ' ', text) 
    return text

data['question'] = data['question'].apply(clean_text)
data['answer'] = data['answer'].apply(clean_text)

print(f">>> Pre-training corpus prepared. Total sequence pairs: {len(data)}")

In [ ]:
# ===========================================================================
# STEP 2-5: Special Token Integration & Tokenizer Training
# ===========================================================================
# To resolve the 'Token Fragmentation' issue, we explicitly register structural 
# delimiters ([User], [Assistant]) as atomic tokens. This ensures the model 
# treats these markers as single semantic units rather than character sequences.
os.environ['GLOG_minloglevel'] = '2'
import sentencepiece as spm

# 1. Exporting for SentencePiece: Saving the cleaned corpus to a flat text file.
with open('./data/corpus.txt', 'w', encoding='utf-8') as f:
    for _, row in data.iterrows():
        f.write(f"[User] {row['question']} [Assistant] {row['answer']}\n")

print(f">>> Filtered corpus created to prevent sequence overflow.")

# 1. Train the Tokenizer
# ===========================================================================
# CRITICAL FIX: Atomic Tokenization for Structural Delimiters
# ===========================================================================
# [Why 'user_defined_symbols' is "Critical"]
# 1. Prevention of Fragmentation: Without this, '[User]' is split into 6 tokens: 
#    '[', 'U', 's', 'e', 'r', ']'. By registering it here, it becomes ONE ID.
# 2. Attention Signal Sharpness: The model's self-attention mechanism can now 
#    perfectly identify the speaker change with 100% confidence in a single step.
# 3. Memory Efficiency: Since we use only 1 token instead of 6 for each tag, 
#    we save 10+ tokens per Q&A pair, allowing for more context in our 'max_len'.
# 4. Phase-2 Alignment: SFT depends on identifying exactly where 'Assistant' 
#    starts. Having a dedicated single token for '[Assistant]' makes masking 
#    logic significantly more robust and less prone to off-by-one errors.
# <s> : Start of Sequence, </s> : End of Sequence
# ===========================================================================
spm.SentencePieceTrainer.train(
    input='./data/corpus.txt',
    model_prefix='./data/spm_chatbot',
    vocab_size=HP["vocab_size"],
    user_defined_symbols=['[User]', '[Assistant]', '<s>', '</s>'], # Critical Fix
    model_type='unigram'
)

# 2. Load the Tokenizer Instance (Critical Step!)
# After training, we must load the .model file into a SentencePieceProcessor 
# instance to perform actual tokenization.
sp = spm.SentencePieceProcessor()
sp.load("./data/spm_chatbot.model")

print(f"\n>>> Stage 0 Complete: Tokenizer trained and loaded.")
print(f">>> Vocabulary Size: {sp.get_piece_size()}")
print(f">>> Special Tokens Check: [User]={sp.piece_to_id('[User]')}, [Assistant]={sp.piece_to_id('[Assistant]')}")

**Technical Rationale for Delimiter Atomicitity:**
> "By defining `[User]` and `[Assistant]` as `user_defined_symbols`, we prevent the tokenizer from breaking them down into sub-character units (e.g., `[`, `U`, `s`, `e`, `r`). This **special token integration** optimizes the self-attention mechanism, as the model can now recognize conversational boundaries through a single embedding vector. This architectural alignment is crucial for minimizing noise during the causal masking process."


##### Tokenizer Refinement: From Fragmentation to Atomicity

* **Initial Problem:**
Structural markers like `[User]` were fragmented into character-level tokens (e.g., `[`, `U`, `s`, `e`, `r`), wasting context window and diluting the role-specific signal.

* **The Fix:**
We transitioned to an **Atomic Tokenization** strategy using SentencePiece’s `user_defined_symbols`. 
  1. **Structural Delimiters**: `[User]` and `[Assistant]` are now mapped to unique IDs (3 and 4), ensuring they are never split.
  2. **Boundary Control**: We integrated `<s>` (BOS) to signal the sequence start.

* **Final Data Structure:**
`<s> [User] {Question} [Assistant] {Answer} </s>`
This layout provides a clear hierarchy between global sequence boundaries (`<s>`, `</s>`) and internal conversational roles (`[User]`, `[Assistant]`).


##### Verification of Token Atomicity

By registering `[User]` and `[Assistant]` as `user_defined_symbols` in SentencePiece, we successfully achieved **Atomic Tokenization**. 

**Technical Advantages:**
1. **Reduced Sequence Overhead**: We saved approximately **83% of the token space** previously wasted on structural delimiters.
2. **Sharper Attention Signal**: The model no longer needs to reconstruct the meaning of a "User" tag from character fragments. It can now directly associate **ID: 3** with the start of a user prompt, leading to faster convergence and higher accuracy.
3. **Consistency**: This ensures that the boundary between human input and AI response is numerically distinct, which is vital for the **Target Masking** process in SFT.

In [ ]:
# ===========================================================================
# Stage 1: Data Preparation 
# ===========================================================================
# def prepare_pretrain_data(data, sp, max_len=128):
#     all_input_ids = []
#     all_labels = []

#     for _, row in data.iterrows():
#         # 1. Sequence Stitching for Causal Language Modeling
#         # In Phase 1, we treat the entire dialogue as a single linguistic stream.
#         full_text = f"<s> [User] {row['question']} [Assistant] {row['answer']} </s>"
        
#         # 2. Tokenization
#         input_ids = sp.encode_as_ids(full_text)[:max_len]
        
#         # 3. Label Creation for Phase 1 (No Masking)
#         # Unlike SFT, Pre-training requires the model to predict EVERY token in the sequence.
#         # We do not use -100 for the prompt; the model learns the entire distribution.
#         labels = input_ids[:] 

#         # 4. Consistent Padding
#         # To maintain uniform tensor shapes for batch processing, we pad up to max_len.
#         # We use ignore_index (-100) only for the padding tokens, but not for the prompts
#         padding_len = max_len - len(input_ids)
#         input_ids += [sp.pad_id() if sp.pad_id() != -1 else 0] * padding_len
#         labels += [-100] * padding_len 

#         all_input_ids.append(input_ids)
#         all_labels.append(labels)

#     return torch.tensor(all_input_ids, dtype=torch.long), torch.tensor(all_labels, dtype=torch.long)
# # Execute preparation for Phase 1
# input_ids, labels = process_pretrain_row(data, sp, max_len=HP['max_len'])
# print(f">>> Processed {len(all_input_ids)} samples.")
# print(">>> Phase 1 Input IDs (First Sample):", all_input_ids[0][:20])
# print(">>> Phase 1 Labels (No Masking except Padding):", all_labels[0][:20])



# ===========================================================================

# Refined Pre-training Preprocessing
def process_pretrain_row(row, sp_model, max_len):
    # Construct continuous text stream with explicit separators
    text = f"{row['question']} [SEP] {row['answer']}"
    tokens = [sp_model.bos_id()] + sp_model.encode_as_ids(text) + [sp_model.eos_id()]
    
    # Truncate to fixed sequence length
    full_seq = tokens[:max_len]
    
    # CLM Objective: Predict the next token for the entire sequence
    labels = full_seq.copy() 
    
    # Dynamic Padding with ignore_index (-100)
    padding_needed = max_len - len(full_seq)
    pad_id = sp_model.pad_id() if sp_model.pad_id() != -1 else 0
    
    full_seq += [pad_id] * padding_needed
    labels += [-100] * padding_needed # Mask padding from loss calculation
    
    return full_seq, labels

# --- STEP 1: Loop through all rows to process the data ---
all_input_ids = []
all_labels = []

# Iterate over the dataframe rows
for _, row in data.iterrows():
    ids, labs = process_pretrain_row(row, sp, max_len=HP['max_len'])
    all_input_ids.append(ids)
    all_labels.append(labs)

# Convert to Tensors for training
input_ids_tensor = torch.tensor(all_input_ids, dtype=torch.long)
labels_tensor = torch.tensor(all_labels, dtype=torch.long)

# --- STEP 2: Verify the output ---
print(f">>> Processed {len(all_input_ids)} samples.")
print(">>> Phase 1 Input IDs (First Sample):", all_input_ids[0][:20])
print(">>> Phase 1 Labels (No Masking except Padding):", all_labels[0][:20])


#### 1. prepare_pretrain_data
##### Technical Differentiation: Pre-training vs. SFT

The fundamental difference between **Phase 1 (Pre-training)** and **Phase 2 (SFT)** lies in the **granularity of the loss calculation** and the **application of the `ignore_index`**.

1. Scope of Supervision
* **Phase 1: Pre-training (Foundation Building)**
    * **Objective**: To learn the underlying linguistic patterns, grammar, and statistical structure of the Korean dialogue corpus.
    * **Masking**: No sequence masking is applied. The model is trained to predict the entire sequence, including the `[User]` prompt and the `[Assistant]` response.
    * **Rationale**: By predicting the prompt itself, the model develops a richer understanding of the conversational context and the distribution of human inquiries.
* **Phase 2: Supervised Fine-Tuning (Instruction Alignment)**
    * **Objective**: To align the model’s behavior specifically toward generating helpful and accurate responses based on user input.
    * **Masking**: The `[User]` prompt is masked using the `-100` index.
    * **Rationale**: Loss is calculated strictly on the assistant's response. This prevents the model from "wasting" gradient updates on learning information it already possesses (the prompt) and focuses its capacity on the **Conditional Probability** of the answer.

1. Utilization of `ignore_index`
While both phases utilize `torch.nn.CrossEntropyLoss(ignore_index=-100)`, the tokens targeted for exclusion differ significantly:

| Feature | Phase 1: Pre-training | Phase 2: SFT |
| :--- | :--- | :--- |
| **Labels used** | Full Sequence (`<s>` to `</s>`) | Response Only |
| **Target Masking** | **None** | **Applied to Prompt** |
| **Role of `-100`** | Exclusively for **Padding Tokens** | **Prompt Tokens + Padding Tokens** |
| **Learning Task** | Unconditional Language Modeling | Goal-oriented Dialogue Modeling |


#### 2. **Rationale for Transitioning to `process_pretrain_row`**

We replaced the initial `process_pretrain_row` function with a more streamlined `prepare_pretrain_data` approach. While we initially considered a simplified `[SEP]`-based approach, we finalized our Phase 1 pipeline using a **Structured Template format** (explicitly labeling Question and Answer). This decision was based on the critical need to improve the success rate of the subsequent **Supervised Fine-Tuning (SFT)** stage.



| Feature | Simplified (`[SEP]`) | Template-based (Q-A Labels) | Advantage for SFT |
| :--- | :--- | :--- | :--- |
| **Instruction Awareness** | Generic boundary. | Explicit Role definition. | Primes the model to understand the **Causal Link** between a User query and an Assistant response. |
| **Format Consistency** | High distribution shift. | Low distribution shift. | Prevents "Catastrophic Forgetting" when moving from Pre-training to SFT. |
| **Syntactic Priming** | Learns general grammar. | Learns **Dialogical Logic**. | Helps the model master not just "what to say," but "how to respond." |


**Why Template-based is Superior to `[SEP]` for SFT Readiness**

1. Preventing "Format Shock" during SFT
The primary reason for SFT failure is often a **Distribution Shift**. If the model learns only the `[SEP]` token during Phase 1 but is suddenly forced to learn `[User]/[Assistant]` headers in Phase 2, it spends the initial SFT epochs unlearning the previous structure. By using a consistent template from Phase 1, we ensure the model's self-attention layers are already tuned to the target dialogue format.

2. Enhancing Structural Priming in Small Datasets
In smaller datasets, unstructured pre-training often fails to provide enough signal for the model to distinguish between "context" and "target." By explicitly marking the segments as Question and Answer, we force the model to prioritize the **Intent-Response relationship**. This makes the SFT stage significantly more effective as the model has already established a foundation of "Role-playing."

3. Solving Convergence Issues
Our experiments showed that the `[SEP]`-only method struggled to reach low validation loss in the SFT stage. The template-based approach provides **clearer gradient signals**. The model learns that certain tokens (like the Answer start marker) are high-entropy transition points, making it much more capable of following instructions during the actual fine-tuning process.


**Comparison of Data Pipelines**

| Feature | Legacy (`prepare_pretrain_data`) | Refined (`process_pretrain_row`) | Improvement |
| :--- | :--- | :--- | :--- |
| **Logic** | Iterates over DataFrame (Batch). | Atomic (Single Row). | Higher scalability and easier debugging. |
| **Token Integrity** | Manual `<s>` strings. | Native `bos_id()`/`eos_id()`. | Prevents sub-token fragmentation and stabilizes attention. |
| **Data Flow** | High Memory Overhead. | Optimized Map-style. | Efficient memory usage during large-scale training. |



In [ ]:
# ===========================================================================
# Stage 2 - Training & Validation Data Split
# ===========================================================================
# We split the dataset into Training (90%) and Validation (10%) sets. 
# This allows us to monitor the 'Generalization Gap' and ensure the model 
# isn't merely memorizing the corpus (Overfitting).
# dataset = TensorDataset(input_ids, labels)

input_ids_tensor = torch.tensor(all_input_ids, dtype=torch.long)
labels_tensor = torch.tensor(all_labels, dtype=torch.long)

dataset = TensorDataset(input_ids_tensor, labels_tensor)

train_size = int(0.9 * len(dataset))
val_size = len(dataset) - train_size

train_dataset, val_dataset = random_split(
    dataset, [train_size, val_size], 
    generator=torch.Generator().manual_seed(42) # 재현성을 위해 시드 고정
)

# ===========================================================================
# Causal Language Modeling DataLoaders
# ===========================================================================
# Batch Size: 64 is a balanced choice for Apple Silicon (MPS) to maximize 
# throughput while maintaining gradient stability.
train_loader = DataLoader(
    train_dataset, 
    batch_size=HP['batch_size'], 
    shuffle=True, 
    drop_last=True # Discard incomplete batches to stabilize gradients
)

val_loader = DataLoader(
    val_dataset, 
    batch_size=HP['batch_size'], 
    shuffle=False
)

print(f">>> DataLoaders initialized.")
print(f">>> Train Batches: {len(train_loader)} | Val Batches: {len(val_loader)}")

In [ ]:
# ===========================================================================
# Stage 3 - Optimization Suite (Phase 1 Calibration)
# ===========================================================================
import torch.optim as optim

# 1. Optimizer Configuration
# Using Adam with Betas tuned for Transformer stability. 
# LR is set as a base scale (1.0), which LambdaLR then modulates.
optimizer = optim.Adam(
    model.parameters(), 
    lr=HP["learning_rate"], 
    betas=(0.9, 0.98), 
    eps=1e-9,
    weight_decay=0.1
)

# # 2. Learning Rate Scheduling (Noam Scheme)
# # The warmup strategy is critical for Phase 1 to prevent gradient explosion 
# # during the early stages of learning the vocabulary distribution.
# lr_lambda_func = get_lr_lambda(HP["d_model"], warmup_steps=HP["warmup_steps"])
# scheduler = lr_scheduler.LambdaLR(optimizer, lr_lambda=lr_lambda_func)

# 3. Objective Function (Cross-Entropy)
# Label Smoothing (0.1) prevents the model from becoming overconfident, 
# encouraging better generalization to unseen dialogue patterns.
criterion = torch.nn.CrossEntropyLoss(
    ignore_index=-100, 
    label_smoothing=HP["label_smoothing"]
)

print(f">>> Optimization suite ready.")
# print(f">>> Scheduler: Noam (Warmup={HP['warmup_steps']} steps)")

In [ ]:
# -------------------------------------------------------------------------
# Stage 4: Pre-training Setup & Smart Resume Logic
# -------------------------------------------------------------------------
# Phase 1 requires longer training; we use separate experiment names to avoid 
# overwriting SFT weights.
START_EPOCH = 1
END_EPOCH = 50
EXP_NAME = "GPT_Pretrain_v1" # Changed to reflect Phase 1

# 1. Resume Logic
if START_EPOCH > 1:
    prev_epoch = START_EPOCH - 1
    print(f">>> [Resume] Resuming Phase 1 from Epoch {prev_epoch}...")
    
    # Pathing convention for Pre-training weights
    resume_path = f"./results/{EXP_NAME}/weights/epoch_{prev_epoch}.pth"
    
    # Load weights into the model
    model = load_weights(model, resume_path, device=device)
    
    # IMPORTANT: Use the best loss from your Phase 1 logs
    # In Pre-training, we aim for the lowest possible cross-entropy loss.
    best_val_loss = 4.5231  # Example placeholder
else:
    print(">>> [Fresh Start] Initializing Phase 1: Foundation Pre-training.")
    best_val_loss = float('inf')

# 2. Early Stopping & Convergence Monitor
# In Pre-training, we often use higher patience than SFT because 
# learning complex linguistic patterns takes more time.
early_stop_counter = 0 
patience = HP.get("patience", 10) # Increased patience for Foundation training
print(f">>> Setup Complete. Experiment: {EXP_NAME} | Patience: {patience}")

#### Training Stability & Checkpoint Persistence

Given the computational intensity of Phase 1 (Pre-training), we implemented a **Robust Resume Logic** to ensure continuity across multi-day training sessions.

1. **State Persistence**: The training pipeline is designed to save model states (weights) at the end of every epoch. The `START_EPOCH` parameter allows the environment to bypass redundant initializations and resume from the specific checkpoint, maintaining the continuity of the **LambdaLR scheduler** state.
2. **Convergence Monitoring**: For Pre-training, we extended the **Early Stopping Patience** to 15 epochs. Unlike SFT, where the model can quickly overfit to specific instructions, Pre-training requires sustained exposure to the corpus to achieve linguistic convergence.
3. **Loss Tracking**: The `best_val_loss` is utilized as a threshold for saving the "Best-So-Far" model, which serves as the starting point for Phase 2 (SFT).

In [ ]:
# --------------------------------------------------
# Stage 5: Main Training Loop
# --------------------------------------------------
import math

print(f"\n>>> Starting Phase 1 (Pre-training): {EXP_NAME} on {device}")
print(f">>> Target Epochs: {START_EPOCH} to {END_EPOCH} | Patience: {HP['patience']}")

for epoch in range(START_EPOCH, END_EPOCH + 1):
    
    # --- PHASE 1: Training (Foundation Building) ---
    # In Phase 1, we focus on minimizing Cross-Entropy across the entire corpus.
    batch_losses, train_acc = train_one_epoch(
        model=model, 
        loader=train_loader, 
        optimizer=optimizer,    
        criterion=criterion,    
        device=device, 
        scheduler=None
    )
    avg_train_loss = sum(batch_losses) / len(batch_losses)
    
    # --- PHASE 2: Validation (Generalization Check) ---
    val_loss, val_acc = validate(model, val_loader, criterion, device)
    
    # [Pre-training Metric] Calculate Perplexity (PPL)
    # PPL measures how "surprised" the model is by the validation data. 
    # Lower is better (closer to 1.0 means perfect prediction).
    val_ppl = math.exp(val_loss) if val_loss < 20 else float('inf')
    
    # --- PHASE 3: Logging & Checkpointing ---
    # Save the current epoch weights for Smart Resume logic.
    save_weights(model, EXP_NAME, HP["learning_rate"], HP["batch_size"], epoch)
    
    # Check for Convergence (Early Stopping Logic)
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        early_stop_counter = 0
        save_weights(model, EXP_NAME, HP["learning_rate"], HP["batch_size"], epoch, is_best=True)
        print(f"   [Update] *** Best Val Loss Improved to {val_loss:.4f} ***")
    else:
        early_stop_counter += 1

    # --- PHASE 4: Console Output (Detailed Metrics) ---
    # We prioritize Loss and PPL over Accuracy for Foundation training.
    print(f"Epoch [{epoch:02d}/{END_EPOCH:02d}] "
          f"Loss: (T) {avg_train_loss:.4f} / (V) {val_loss:.4f} | "
          f"PPL: {val_ppl:.2f} | "
          f"LR: {optimizer.param_groups[0]['lr']:.7f}")

    # --- PHASE 5: Early Stopping ---
    if early_stop_counter >= HP["patience"]:
        print(f"\n>>> [Termination] No improvement for {HP['patience']} epochs.")
        print(f">>> Phase 1 training halted. Best Val Loss: {best_val_loss:.4f}")
        break

print("\n>>> Phase 1 Foundation Training Completed.")

**1. Why does the "Best" appear so early?**

In deep learning, an early convergence (like Epoch 4) usually happens due to the following technical reasons:

* **High Information Density vs. Small Dataset**: 
  * If your pre-training dataset is relatively small or repetitive, the model can "scan" all the fundamental patterns within just a few passes.
  * Once it understands the basic structure, additional epochs only lead to memorizing specific noise.
* **Optimal Learning Rate ($5 \times 10^{-5}$)**: 
  * You found a very stable learning rate. Unlike higher rates that "bounce" around the target, this rate allowed the model to slide directly into a good local minimum quickly.
* **Model Capacity**: 
  * Your Transformer architecture (e.g., 4 layers) is powerful enough to represent this specific data distribution almost immediately.

In [ ]:
import torch
import sentencepiece as spm

# After Restart
# --- STEP 1: Initialize Environment and Tokenizer ---
if torch.backends.mps.is_available():
    device = torch.device("mps")
    print(">>> Using Apple Silicon (MPS) for acceleration.")
else:
    device = torch.device("cpu")
    print(">>> Using CPU.")
    
# Load the SentencePieceProcessor object (NOT just the string path)
sp = spm.SentencePieceProcessor()
sp.load("./data/spm_chatbot.model")


# --- STEP 2: Re-initialize Model Architecture ---
# Note: You must have your Model class defined (e.g., GPT)
# model = GPT(vocab_size=len(sp), ...) 
model.to(device)

# --- STEP 3: Load the Best Weights (Epoch 04) ---
# This allows you to skip training and go straight to testing
model_path = "results/GPT_Pretrain_v1/weights/best_model.pth"

try:
    # Use map_location to ensure it loads correctly regardless of the original device
    checkpoint = torch.load(model_path, map_location=device)
    model.load_state_dict(checkpoint)
    print(f"[Success] Loaded 'Best' weights (Epoch 04) from {model_path}")
except FileNotFoundError:
    print(f"[Error] Weight file not found at {model_path}. Check your directory structure.")

# --- STEP 4: Execute Test with English Comments ---
print("\n" + "="*12 + " GPT Complex Template Baseline Test " + "="*12)

# Ensure evaluation mode is on
model.eval() 

test_queries = [
    # Basic conversation
    "안녕",
    "고마워",
    "미안해",
    "잘 지냈어?",
    "오늘 뭐해?",

    # General questions (check generalization)
    "오늘 날씨 어때?",
    "점심 뭐 먹을까?",
    "주말에 뭐 하지?",

    # Identity / role consistency
    "너 누구야?",
    "너 뭐하는 애야?"
]

with torch.no_grad():
    for q in test_queries:
        # RATIONALE: Using the complex template to trigger the model's role-playing logic
        input_prompt = f"<s> [User] {q} [Assistant]"
        
        # FIXED CALL: Ensure the order matches your function (model, sp, prompt)
        answer = greedy_decode_gpt(model=model, sp=sp, prompt=input_prompt, device=device, max_len=64)
        
        print(f"User Query: {q}")
        print(f"Model Response: {answer}\n")

print("="*50)

**Technical Post-Mortem: Parroting and Recursive Loop**

The baseline test shows the model "copying" input tokens or repeating structural tags (`[Assistant]`, `</s>`).

**Technical Breakdown:**
1. **Low Semantic Entropy**: In Phase 1, the model learned the **syntax** (how a sentence looks) but not the **semantics** (what a sentence means). Without SFT, the model lacks the "Instruction-Following" capability, leading to a simple repetition of the most frequent patterns it saw: the structural tags.
2. **Greedy Decoding Bottleneck**: Since we are using `greedy_decode`, the model consistently picks the single most probable token. In a pre-trained-only state, structural markers often have an overwhelming probability, trapping the model in a recursive loop.
3. **The "Empty Vessel" State**: The model has successfully built the "Dialogue Container" (knowing where [User] and [Assistant] go), but the container is empty. This is the **Perfect Starting Point** for SFT, as we can now fill this container with specific response data.